In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when, desc
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Pertemuan7-StudiKasusTerintegrasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [5]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/maulzz/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/maulzz/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/maulzz/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/maulzz/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [14]:
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/maulzz/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/maulzz/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/maulzz/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

print("--- A. Extract Berhasil ---")

--- A. Extract Berhasil ---


In [8]:
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
    .join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

print("\n--- B. Hasil Join dan Kolom Turunan (5 Baris Pertama) ---")
df_gabungan.show(5)


--- B. Hasil Join dan Kolom Turunan (5 Baris Pertama) ---
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusan

In [9]:
df_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu") \
    .agg(spark_sum("total_penjualan").alias("penjualan_menu"))

window_cabang = Window.partitionBy("nama_cabang").orderBy(desc("penjualan_menu"))

df_top2_menu = df_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
    .filter(col("peringkat") <= 2)

print("\n--- C. Top-2 Menu Terlaris per Cabang ---")
df_top2_menu.show()


--- C. Top-2 Menu Terlaris per Cabang ---
+--------------------+---------+--------------+---------+
|         nama_cabang|nama_menu|penjualan_menu|peringkat|
+--------------------+---------+--------------+---------+
|Kopi Nusantara Ma...|   Menu-1|       8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|       8085000|        2|
|Kopi Nusantara Se...|  Menu-13|       8750000|        1|
|Kopi Nusantara Se...|   Menu-9|       8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|       9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|       8160000|        2|
+--------------------+---------+--------------+---------+



In [10]:
df_gabungan.createOrReplaceTempView("tabel_kopi")

kueri_barista = """
SELECT 
    kepala_barista, 
    SUM(total_penjualan) AS total_pendapatan
FROM tabel_kopi
GROUP BY kepala_barista
ORDER BY total_pendapatan DESC
"""

print("\n--- D. Total Penjualan per Kepala Barista (Spark SQL) ---")
spark.sql(kueri_barista).show()


--- D. Total Penjualan per Kepala Barista (Spark SQL) ---
+--------------+----------------+
|kepala_barista|total_pendapatan|
+--------------+----------------+
|         Nadia|       177680000|
|          Reza|        84645000|
+--------------+----------------+



In [12]:
path_simpan_tugas = "hdfs://localhost:9000/user/maulzz/tugas7/processed/laporan"

df_gabungan.write \
    .mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(path_simpan_tugas)

df_verifikasi = spark.read.parquet(path_simpan_tugas)
jumlah_sebelum = df_gabungan.count()
jumlah_sesudah = df_verifikasi.count()

print(f"\n--- E. Verifikasi Load HDFS ---")
print(f"Jumlah baris sebelum disimpan: {jumlah_sebelum}")
print(f"Jumlah baris tersimpan: {jumlah_sesudah}")
if jumlah_sebelum == jumlah_sesudah:
    print("Status: Berhasil (Tidak ada data hilang)")


--- E. Verifikasi Load HDFS ---
Jumlah baris sebelum disimpan: 4000
Jumlah baris tersimpan: 4000
Status: Berhasil (Tidak ada data hilang)


In [15]:
from pyspark.sql.functions import count, avg

ringkasan_tugas = df_gabungan.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(desc("total_penjualan"))

print("\n--- F. Ringkasan Eksekutif ---")
ringkasan_tugas.show()


--- F. Ringkasan Eksekutif ---
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+

